# Weighted sampler and pos_weight


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from core.cli import data_module
from core.dataset import sample_weights
from helpers.config import CLASSES, ROOT, SEED
from helpers.plots import save_figures, use_style

CONFIG = ROOT / "configs" / "exp34_convnext_tiny" / "convnext_tiny.yaml"
FIGURES = ROOT / "docs" / "figures"
N_BATCHES = 4

use_style()

## Training rows of fold 0


In [ ]:
dm = data_module(CONFIG, 0)
dm.setup()

train = dm.train_ds.df
labels = train[CLASSES].to_numpy()
batch_size = dm.batch_size

len(train), batch_size

## Batches


In [ ]:
torch.manual_seed(SEED)

# without sampler: plain shuffle
plain = list(torch.utils.data.RandomSampler(train))

# with sampler: same as `KaggleDataModule.train_dataloader`
weights = sample_weights(train, dm.oversample, dm.downsample)
sampler = torch.utils.data.WeightedRandomSampler(
    weights, num_samples=round(weights.sum()), replacement=True
)
weighted = list(sampler)

n = N_BATCHES * batch_size
plain_batches = np.array(plain[:n]).reshape(N_BATCHES, batch_size)
weighted_batches = np.array(weighted[:n]).reshape(N_BATCHES, batch_size)

# class counts per batch, shape (N_BATCHES, len(CLASSES))
plain_counts = labels[plain_batches].sum(axis=1)
weighted_counts = labels[weighted_batches].sum(axis=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
x = np.arange(N_BATCHES)
width = 0.8 / len(CLASSES)

for ax, counts, title in zip(
    axes, [plain_counts, weighted_counts], ["Without sampler", "With sampler"]
):
    for i, cls in enumerate(CLASSES):
        ax.bar(x + i * width, counts[:, i], width, label=cls)

    ax.set(title=title, xlabel="batch", xticks=x + 0.4 - width / 2, xticklabels=x + 1)

axes[0].set_ylabel(f"rows with label (batch of {batch_size})")
axes[1].legend()
fig.tight_layout()

save_figures({"wsampler_posweight_batches.png": fig}, FIGURES)

## Expected vs actual


In [ ]:
n_pos = labels.sum(axis=0)
draws_pos = weights @ labels

table = pd.DataFrame(
    {
        "expected per batch, no sampler": batch_size * n_pos / len(train),
        "actual per batch, no sampler": plain_counts.mean(axis=0),
        "expected per batch, sampler": batch_size * draws_pos / weights.sum(),
        "actual per batch, sampler": weighted_counts.mean(axis=0),
        "pos_weight": dm.pos_weight,
    },
    index=CLASSES,
)

table.round(2)